# Week 14: Final Project — Fast Log Indexer
## PHASE 7: Proving Mastery

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Apply data structures and algorithms learned throughout the course to a real-world project
2. Generate and work with realistic log data
3. Build **indexes** using dictionaries and sets for fast lookups
4. Implement **fast queries** by time range, by ID, and by threshold
5. **Benchmark** indexed queries against brute-force scanning
6. Create **visualizations** showing the performance difference
7. Write a short conclusion explaining why your data structure choices were correct

## 🎯 Core Mastery Connection

This is where you prove mastery. You will choose data structures (dict, sorted list, set), implement indexed queries, benchmark them against brute force, and demonstrate — with empirical evidence — that your choices were right. The entire course has built toward this moment: choose, implement, measure, and prove.

---
## 🧭 Three-Hour Interactive Studio Plan

**Audience:** Mechatronics Engineering students  
**Weekly focus:** Week 14: Final Project — Fast Log Indexer

**Professional lens:** building a fast, explainable index for engineering log data.

| Time | Learning cycle |
|---|---|
| 00:00–00:10 | Launch question, prior-knowledge retrieval, outcomes |
| 00:10–00:50 | Concept cycle 1: explain → predict → test |
| 00:50–01:00 | Checkpoint 1, student questions, peer explanation |
| 01:00–01:10 | Break |
| 01:10–01:50 | Concept cycle 2: worked example → variation → discussion |
| 01:50–02:00 | Checkpoint 2 and misconception repair |
| 02:00–02:10 | Break |
| 02:10–02:40 | Core in-class practice with instructor circulation |
| 02:40–02:50 | Checkpoint 3: exam bridge and professional transfer |
| 02:50–03:00 | Open questions, summary, and exit ticket |

The official start and finish times are followed as published in the timetable. Ask questions at any point; the scheduled checkpoints guarantee additional question time. Checkpoints are private self-checks in this runtime—no identity, upload, homework, or instructor dashboard.


In [ ]:
# Run once. This pulse stays only in the current Colab runtime.
_studio_pulses = {}

def studio_pulse(number, response, minimum_words=8):
    words = str(response).strip().split()
    ready = len(words) >= minimum_words
    _studio_pulses[int(number)] = ready
    if ready:
        print(f"✅ Checkpoint {number}: explanation recorded locally ({len(words)} words).")
    else:
        print(f"🟡 Checkpoint {number}: explain your reasoning in at least {minimum_words} words, then retry.")
    print("Nothing is transmitted or stored for grading.")
    return ready

print("✅ Local studio checkpoints ready")

---
## Part 1: Project Overview

### The Scenario

You are a junior developer at a tech company. Your team receives **thousands of log records** from servers every day. Each log record contains:

| Field | Type | Description | Example |
|-------|------|-------------|--------|
| `timestamp` | str | When the event happened | `"2024-03-14 10:30:45"` |
| `id` | str | Which server/device produced it | `"server-042"` |
| `level` | str | Severity level | `"INFO"`, `"WARNING"`, `"ERROR"` |
| `message` | str | What happened | `"Request processed"` |
| `value` | float | A numeric metric (e.g., response time) | `0.245` |

### The Problem

Searching through logs by scanning every record is **slow**. Your task is to build **indexes** (using `dict` and `set`) that make common queries fast.

### Real-World Analogy: Library Card Catalog 📚

| Without Index | With Index |
|--------------|------------|
| Walk through every shelf to find a book | Look up the card catalog by author/title |
| O(n) — check every book | O(1) — jump straight to the right shelf |

### Project Structure

The project has **6 parts** (each is an exercise):

1. **EX1:** Generate sample log data
2. **EX2:** Build indexes using dict/set
3. **EX3:** Implement fast queries (by time range)
4. **EX4:** Implement fast queries (by ID and threshold)
5. **EX5:** Benchmark indexed vs brute-force
6. **EX6:** Visualize results and write conclusion

---
### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

For **Week 14: Final Project — Fast Log Indexer**, state the key invariant, operation cost, or decision rule in your own words.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_1_response = ""  # write at least 8 words
studio_pulse(1, checkpoint_1_response)

---
## Part 2: Starter Code & Data Generation

First, let's understand the log data format and provide helper utilities.

**Figure 2.1** — Log record structure and sample data

In [ ]:
# A single log record is a dictionary with these fields:
sample_record = {
    "timestamp": "2024-03-14 10:30:45",
    "id": "server-042",
    "level": "INFO",
    "message": "Request processed successfully",
    "value": 0.245
}

print("Sample Log Record:")
for key, val in sample_record.items():
    print(f"  {key:<12} : {val}")

**Figure 2.2** — Helper: Log data generator (provided for you)

In [ ]:
import random
import time
from datetime import datetime, timedelta

def generate_logs(n=10000, seed=42):
    """Generate n sample log records.
    
    Returns a list of dictionaries, each with:
        timestamp (str), id (str), level (str), message (str), value (float)
    """
    random.seed(seed)
    
    server_ids = [f"server-{i:03d}" for i in range(1, 51)]  # 50 servers
    levels = ["DEBUG", "INFO", "INFO", "INFO", "WARNING", "WARNING", "ERROR"]  # INFO most common
    messages = [
        "Request processed",
        "Connection established",
        "Cache miss",
        "Database query executed",
        "File uploaded",
        "User logged in",
        "Session timeout",
        "Memory usage high",
        "Disk space low",
        "Service restarted",
        "API rate limit reached",
        "Backup completed",
    ]
    
    base_time = datetime(2024, 1, 1, 0, 0, 0)
    logs = []
    
    for i in range(n):
        # Random timestamp within 90 days
        offset_seconds = random.randint(0, 90 * 24 * 3600)
        ts = base_time + timedelta(seconds=offset_seconds)
        
        level = random.choice(levels)
        # Higher values for errors
        if level == "ERROR":
            value = round(random.uniform(5.0, 30.0), 3)
        elif level == "WARNING":
            value = round(random.uniform(1.0, 10.0), 3)
        else:
            value = round(random.uniform(0.01, 3.0), 3)
        
        logs.append({
            "timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
            "id": random.choice(server_ids),
            "level": level,
            "message": random.choice(messages),
            "value": value,
        })
    
    # Sort by timestamp
    logs.sort(key=lambda x: x["timestamp"])
    return logs

# Generate the dataset
logs = generate_logs(10000)

print(f"Generated {len(logs)} log records.")
print(f"\nFirst 5 records:")
for i, log in enumerate(logs[:5]):
    print(f"  [{i}] {log['timestamp']}  {log['id']:<12} {log['level']:<8} {log['message']:<30} value={log['value']}")

print(f"\nLast 3 records:")
for i, log in enumerate(logs[-3:], len(logs)-3):
    print(f"  [{i}] {log['timestamp']}  {log['id']:<12} {log['level']:<8} {log['message']:<30} value={log['value']}")

**Figure 2.3** — Quick data summary

In [ ]:
# Let's understand our data
level_counts = {}
server_counts = {}
values = []

for log in logs:
    level_counts[log["level"]] = level_counts.get(log["level"], 0) + 1
    server_counts[log["id"]] = server_counts.get(log["id"], 0) + 1
    values.append(log["value"])

print("Log Level Distribution:")
for level, count in sorted(level_counts.items()):
    bar = "█" * (count // 50)
    print(f"  {level:<10} {count:>5}  {bar}")

print(f"\nUnique Servers: {len(server_counts)}")
print(f"Value Range: {min(values):.3f} to {max(values):.3f}")
print(f"Average Value: {sum(values)/len(values):.3f}")
print(f"Time Range: {logs[0]['timestamp']} to {logs[-1]['timestamp']}")

---
## Part 3: Brute-Force Queries (The Slow Way)

Before building indexes, let's see how brute-force scanning works. This is the approach you'd use **without** any data structures knowledge.

**Figure 3.1** — Brute-force query implementations

In [ ]:
def brute_force_by_level(logs, level):
    """Find all logs with a specific level by scanning everything."""
    results = []
    for log in logs:
        if log["level"] == level:
            results.append(log)
    return results

def brute_force_by_id(logs, server_id):
    """Find all logs from a specific server by scanning everything."""
    results = []
    for log in logs:
        if log["id"] == server_id:
            results.append(log)
    return results

def brute_force_by_time_range(logs, start_time, end_time):
    """Find all logs in a time range by scanning everything."""
    results = []
    for log in logs:
        if start_time <= log["timestamp"] <= end_time:
            results.append(log)
    return results

def brute_force_by_threshold(logs, min_value):
    """Find all logs with value >= threshold by scanning everything."""
    results = []
    for log in logs:
        if log["value"] >= min_value:
            results.append(log)
    return results

# Quick test
errors = brute_force_by_level(logs, "ERROR")
print(f"Brute-force found {len(errors)} ERROR logs")

server_logs = brute_force_by_id(logs, "server-001")
print(f"Brute-force found {len(server_logs)} logs from server-001")

time_logs = brute_force_by_time_range(logs, "2024-02-01 00:00:00", "2024-02-28 23:59:59")
print(f"Brute-force found {len(time_logs)} logs in February")

high_val = brute_force_by_threshold(logs, 10.0)
print(f"Brute-force found {len(high_val)} logs with value >= 10.0")

---
### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

Predict what happens when the input size doubles. Justify the trend with an operation count or complexity class—not timing alone.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_2_response = ""  # write at least 8 words
studio_pulse(2, checkpoint_2_response)

---
## Part 4: Building Indexes (The Fast Way)

Now let's see how to build indexes. An **index** is a data structure that maps a **key** (like server ID or log level) to the **positions** of matching records.

| Index Type | Key | Value | Data Structure | Lookup Time |
|-----------|-----|-------|----------------|-------------|
| By Level | `"ERROR"` | List of log indices | `dict[str, list]` | O(1) |
| By Server ID | `"server-042"` | List of log indices | `dict[str, list]` | O(1) |
| By Date | `"2024-02-14"` | List of log indices | `dict[str, list]` | O(1) |
| By Value | threshold | Sorted list of (value, index) | sorted list | O(log n) |

**Figure 4.1** — Example: Building a level index

In [ ]:
# EXAMPLE: Build an index by log level
# This maps each level to a list of indices into the logs list

level_index = {}  # key: level string, value: list of log indices

for i, log in enumerate(logs):
    level = log["level"]
    if level not in level_index:
        level_index[level] = []
    level_index[level].append(i)

print("Level Index Built!")
print(f"\nIndex contents:")
for level, indices in sorted(level_index.items()):
    print(f"  {level:<10}: {len(indices)} records (first 5 indices: {indices[:5]})")

# Now querying is instant!
print(f"\n--- Fast Query Example ---")
error_indices = level_index.get("ERROR", [])
print(f"ERROR logs: {len(error_indices)} found instantly (O(1) lookup!)")
print(f"First 3 ERROR logs:")
for idx in error_indices[:3]:
    log = logs[idx]
    print(f"  [{idx}] {log['timestamp']} {log['id']} value={log['value']}")

**Figure 4.2** — Example: Building a date index

In [ ]:
# EXAMPLE: Build an index by date (extract date from timestamp)

date_index = {}  # key: date string "YYYY-MM-DD", value: list of log indices

for i, log in enumerate(logs):
    date = log["timestamp"][:10]  # Extract "YYYY-MM-DD" from timestamp
    if date not in date_index:
        date_index[date] = []
    date_index[date].append(i)

print(f"Date Index Built! {len(date_index)} unique dates.")
print(f"\nSample dates and their log counts:")
sample_dates = sorted(date_index.keys())[:5]
for date in sample_dates:
    print(f"  {date}: {len(date_index[date])} logs")

# Fast date range query using the index
def indexed_by_date_range(logs, date_index, start_date, end_date):
    """Query logs by date range using the date index."""
    results = []
    for date in sorted(date_index.keys()):
        if start_date <= date <= end_date:
            for idx in date_index[date]:
                results.append(logs[idx])
    return results

feb_logs = indexed_by_date_range(logs, date_index, "2024-02-01", "2024-02-28")
print(f"\nFast query: {len(feb_logs)} logs in February 2024")

### Quick Recap: `bisect` Module (from Week 5)

The `bisect` module provides efficient binary search functions for sorted lists. We first used it in **Week 5** (Searching). Here is a quick refresher before we use it for threshold queries.

In [ ]:
import bisect

sorted_list = [10, 20, 30, 40, 50, 60, 70]

# bisect_left: index where value would be inserted (leftmost position)
pos_left = bisect.bisect_left(sorted_list, 35)
print(f"bisect_left({sorted_list}, 35)  = {pos_left}   # 35 would go before index {pos_left}")

# bisect_right: index where value would be inserted (rightmost position)
pos_right = bisect.bisect_right(sorted_list, 30)
print(f"bisect_right({sorted_list}, 30) = {pos_right}   # after the existing 30")

# Practical use: find all elements >= 35
print(f"\nElements >= 35: {sorted_list[pos_left:]}")  # slice from insertion point onward

**Figure 4.3** — Using `bisect` for threshold queries

In [ ]:
import bisect

# EXAMPLE: Build a sorted value index for threshold queries
# Store (value, index) pairs sorted by value

value_index = sorted((log["value"], i) for i, log in enumerate(logs))

print(f"Value Index Built! {len(value_index)} entries, sorted by value.")
print(f"\nSmallest 3 values: {[v for v, i in value_index[:3]]}")
print(f"Largest 3 values:  {[v for v, i in value_index[-3:]]}")

# Fast threshold query using binary search
def indexed_by_threshold(logs, value_index, min_value):
    """Find all logs with value >= min_value using binary search."""
    # Find the position where min_value would be inserted
    pos = bisect.bisect_left(value_index, (min_value,))
    # Everything from pos onward has value >= min_value
    return [logs[i] for v, i in value_index[pos:]]

high_val = indexed_by_threshold(logs, value_index, 10.0)
print(f"\nFast query: {len(high_val)} logs with value >= 10.0")

---
## Part 5: Common Errors When Building Indexes

**Figure 5.1** — Common Error: KeyError when index key doesn't exist

In [ ]:
# ERROR: Accessing a key that doesn't exist in the index
print("=" * 50)
print("ERROR: Missing key in index")
print("=" * 50)

test_index = {"INFO": [0, 1, 2], "ERROR": [3, 4]}

try:
    result = test_index["CRITICAL"]  # This key doesn't exist!
except KeyError as e:
    print(f"\n❌ KeyError: {e}")
    print("\n💡 Fix: Use .get() with a default value:")
    result = test_index.get("CRITICAL", [])  # Returns empty list
    print(f"   test_index.get('CRITICAL', []) = {result}")

**Figure 5.2** — Common Error: Storing records instead of indices

In [ ]:
# ERROR: Storing full records wastes memory
print("=" * 50)
print("ERROR: Storing full records vs indices")
print("=" * 50)

import sys

# BAD: storing full records (duplicates data)
bad_index = {}
for log in logs[:1000]:
    level = log["level"]
    if level not in bad_index:
        bad_index[level] = []
    bad_index[level].append(log)  # Stores the whole dict!

# GOOD: storing just indices
good_index = {}
for i, log in enumerate(logs[:1000]):
    level = log["level"]
    if level not in good_index:
        good_index[level] = []
    good_index[level].append(i)  # Stores just the integer index!

bad_size = sys.getsizeof(bad_index) + sum(sys.getsizeof(v) for v in bad_index.values())
good_size = sys.getsizeof(good_index) + sum(sys.getsizeof(v) for v in good_index.values())

print(f"\n  BAD (stores records):  ~{bad_size:,} bytes")
print(f"  GOOD (stores indices): ~{good_size:,} bytes")
print(f"\n💡 Store indices, not copies of data!")

---
## 🎢 Exercises (Project Parts)

Complete all 6 exercises below to build your Fast Log Indexer. Each exercise is one part of the project.

The exercises are structured as:
- **Easy (4):** EX1–EX4 — Generate data and build indexes
- **Medium (6):** EX5–EX10 — Implement queries and benchmarks
- **Challenge (2):** EX11–EX12 — Visualization and conclusion

### Easy Exercises

**EX1 (Easy):** Generate log data using the provided `generate_logs()` function. Generate **20,000** logs. Print the total count, the first 3 records, and the count of each log level.

Expected Output (format):
```
Total logs: 20000
First 3 records:
  [0] 2024-01-01 00:01:23  server-XXX  INFO     ...
  [1] 2024-01-01 00:02:45  server-XXX  WARNING  ...
  [2] 2024-01-01 00:03:12  server-XXX  INFO     ...
Level counts:
  DEBUG   : XXXX
  INFO    : XXXX
  WARNING : XXXX
  ERROR   : XXXX
```

<details><summary>💡 Hint</summary>
Call generate_logs(20000). Loop through to count levels using a dictionary. Print formatted output.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 (Easy):** Build an **ID index** — a dictionary that maps each server ID to a list of log indices. Print how many unique servers there are and the top 5 servers by log count.

Expected Output (format):
```
ID Index built with XX unique servers.
Top 5 servers by log count:
  server-XXX: XXX logs
  server-XXX: XXX logs
  ...
```

<details><summary>💡 Hint</summary>
Loop through logs with enumerate(). For each log, add the index i to id_index[log["id"]]. Use sorted() with a key to find the top 5.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 (Easy):** Build a **level index** — a dictionary that maps each log level to a list of log indices. Also build a **date index** that maps each date string ("YYYY-MM-DD") to a list of log indices. Print the sizes of each index.

Expected Output (format):
```
Level Index: X levels
  DEBUG   : XXXX logs
  INFO    : XXXX logs
  WARNING : XXXX logs
  ERROR   : XXXX logs

Date Index: XX unique dates
  First date: 2024-01-01 (XXX logs)
  Last date:  2024-03-31 (XXX logs)
```

<details><summary>💡 Hint</summary>
For the date index, extract the date from timestamp using log["timestamp"][:10]. Build both indexes in a single loop through the logs.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 (Easy):** Build a **sorted value index** for threshold queries. Create a sorted list of `(value, index)` tuples. Print the 5 smallest and 5 largest values with their server IDs.

Expected Output (format):
```
Value Index built: XXXXX entries (sorted)

5 smallest values:
  value=0.010  server-XXX  INFO
  ...

5 largest values:
  value=29.XXX  server-XXX  ERROR
  ...
```

<details><summary>💡 Hint</summary>
Use a list comprehension: sorted((log["value"], i) for i, log in enumerate(logs)). Access the original log via logs[index] to get server ID and level.
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium Exercises

**EX5 (Medium):** Write an **indexed query function** `fast_query_by_id(logs, id_index, server_id)` that uses your ID index to find all logs from a given server. Compare its output with the brute-force version to verify correctness.

Expected Output (format):
```
Fast query for server-010: XXX logs found
Brute-force for server-010: XXX logs found
Results match: True
```

<details><summary>💡 Hint</summary>
Look up id_index.get(server_id, []) to get the list of indices, then return [logs[i] for i in indices]. Compare lengths and spot-check a few records.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 (Medium):** Write an **indexed query function** `fast_query_by_time_range(logs, date_index, start_date, end_date)` that uses your date index to find logs in a date range. Compare with brute-force.

Expected Output (format):
```
Fast query for Feb 2024: XXX logs found
Brute-force for Feb 2024: XXX logs found
Results match: True
```

<details><summary>💡 Hint</summary>
Loop through sorted date_index keys. If start_date <= date <= end_date, collect all logs at those indices. For brute-force comparison, make sure to compare just the date portion.
</details>

In [ ]:
# ✏️ [EX6] Your code here


---
### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

Choose one completed core exercise. Explain why the algorithm is correct, its dominant cost, and one mechatronics situation where that cost matters.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_3_response = ""  # write at least 8 words
studio_pulse(3, checkpoint_3_response)

---
## 🌟 Optional Extension

Exercises 7 and above are enrichment for remaining class time or independent curiosity. They are not homework and are not collected.


**EX7 (Medium):** Write an **indexed threshold query** `fast_query_by_threshold(logs, value_index, min_value)` using `bisect.bisect_left` on the sorted value index. Compare with brute-force.

Expected Output (format):
```
Fast query for value >= 10.0: XXX logs found
Brute-force for value >= 10.0: XXX logs found
Results match: True
```

<details><summary>💡 Hint</summary>
Use bisect.bisect_left(value_index, (min_value,)) to find the starting position. Everything from that position onward has value >= min_value.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 (Medium):** Write a **combined query** `fast_query_errors_high_value(logs, level_index, value_index, min_value)` that finds all ERROR logs with value >= min_value. Use set intersection of index results for speed. Compare with brute-force.

Expected Output (format):
```
ERROR logs with value >= 15.0: XX logs found
Brute-force: XX logs found
Results match: True
```

<details><summary>💡 Hint</summary>
Get the set of ERROR indices from level_index. Get the set of high-value indices from value_index (using bisect). Intersect the two sets. Return logs at those indices.
</details>

**EX9 (Medium):** **Benchmark** your indexed queries vs brute-force. For each query type (by ID, by time range, by threshold), measure the time for 100 repeated queries. Print a comparison table.

Expected Output (format):
```
Benchmark Results (100 queries each):

Query Type         Brute-Force (s)   Indexed (s)    Speedup
--------------------------------------------------------------
By Server ID       X.XXXXXX          X.XXXXXX       XXx faster
By Time Range      X.XXXXXX          X.XXXXXX       XXx faster
By Threshold       X.XXXXXX          X.XXXXXX       XXx faster
```

<details><summary>💡 Hint</summary>
Use time.perf_counter() before and after a loop of 100 queries. Calculate speedup as brute_time / indexed_time. Use the same query parameters for fair comparison.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 (Medium):** **Scalability test.** Generate logs of different sizes (1000, 5000, 10000, 20000, 50000) and measure brute-force vs indexed query time for each size. Store the results in lists for plotting in EX11.

Expected Output (format):
```
Scalability Test (by-ID query, 50 repetitions each):

Log Size    Brute-Force (s)   Indexed (s)    Speedup
------------------------------------------------------
1000        X.XXXXXX          X.XXXXXX       Xx
5000        X.XXXXXX          X.XXXXXX       Xx
10000       X.XXXXXX          X.XXXXXX       Xx
20000       X.XXXXXX          X.XXXXXX       Xx
50000       X.XXXXXX          X.XXXXXX       Xx
```

<details><summary>💡 Hint</summary>
Loop through each size: generate logs, build the index, then benchmark both approaches. Store brute_times and indexed_times in lists for plotting.
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge Exercises

**EX11 (Challenge):** Create **two plots** using matplotlib:

1. **Plot 1:** Line chart showing brute-force time vs indexed time for increasing log sizes (from EX10 data). Both lines on the same axes.
2. **Plot 2:** Bar chart showing the speedup factor for each log size.

Use proper labels, titles, legends, and grid. Use `fig, axes = plt.subplots(1, 2, figsize=(14, 5))`.

<details><summary>💡 Hint</summary>
Use the lists from EX10 (sizes, brute_times, indexed_times). For plot 1, use axes[0].plot(). For plot 2, use axes[1].bar(). Add xlabel, ylabel, title, legend, and grid.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 (Challenge):** Write a short **conclusion paragraph** (as a Python print statement) that answers these questions:

1. Which data structures did you use for indexing and why?
2. What was the approximate speedup you observed?
3. How did the speedup change as the data size increased?
4. In what real-world scenarios would this matter?

Print your conclusion as a nicely formatted multi-line string.

<details><summary>💡 Hint</summary>
Reference your actual benchmark numbers. Mention dict (O(1) lookup), sorted list + bisect (O(log n) search). Explain that brute-force is O(n) per query while indexed is O(1) or O(log n).
</details>

In [ ]:
# ✏️ [EX12] Your code here


---
## 📚 Course Recap: 14 Weeks at a Glance

Congratulations on completing the entire course! Here is a summary of everything you have learned:

| Week | Topic | Key Concepts |
|------|-------|------|
| 1 | Big-O Notation & Benchmarking | Time/space complexity, O(1), O(n), O(n log n), O(n²), measuring performance |
| 2 | Python Built-in Data Structure Costs | list, dict, set operation costs, amortized analysis |
| 3 | Stack, Queue, Deque (ADTs) | LIFO, FIFO, deque, real-world applications |
| 4 | Recursion & Memoization | Base cases, recursive thinking, caching results |
| 5 | Searching (Linear & Binary Search) | Linear search, binary search, divide and conquer |
| 6 | Sorting Algorithms | Insertion sort, merge sort, quick sort |
| 7 | Hashing & Hash Tables | Hash functions, collisions, dict internals |
| 8 | Heaps & Priority Queues (heapq) | Min/max heaps, heapq module, priority scheduling |
| 9 | Binary Trees & Traversals | Tree structure, in-order, pre-order, post-order, level-order |
| 10 | Binary Search Trees (BST) | Insert, search, delete, balanced vs unbalanced |
| 11 | Graphs (BFS, DFS) | Adjacency lists, breadth-first, depth-first search |
| 12 | Dijkstra's Algorithm | Shortest paths, weighted graphs, priority queue usage |
| 13 | Dynamic Programming | Memoization, tabulation, classic DP problems |
| 14 | Final Project | Fast Log Indexer — applying everything together |

---
## 🎓 Course Complete

Congratulations on completing **Data Structures & Algorithms**!

Over 14 weeks, you have built a strong foundation in:

- **Data Structures:** Lists, dictionaries, sets, stacks, queues, heaps, trees, graphs
- **Algorithms:** Searching, sorting, hashing, graph traversal, dynamic programming
- **Analysis:** Big-O complexity, benchmarking, comparing approaches
- **Problem Solving:** Breaking down problems, choosing the right data structure, optimizing solutions

These skills will serve you well in software development, technical interviews, and any field that involves working with data. Keep practicing, keep building, and keep learning!

**Thank you for a great semester!** 🌟